# 🧲 Геомагнитный ИИ-агент — запуск в Google Colab

Автономный геофизический агент: **Qwen 2.5 7B Instruct** работает локально на
бесплатном GPU Tesla T4 и вызывает инструменты этапов 1–2
(загрузчик INTERMAGNET, анализатор, Plotly-графики).

Никаких платных API и ключей. Модель скачивается один раз с HuggingFace
(~4.7 ГБ) и остаётся в `/content`.

---

## 🚀 Как запустить

| # | Действие |
|---|---|
| 1 | **Среда выполнения → Сменить среду выполнения → GPU (T4)** |
| 2 | Запускайте ячейки **сверху вниз**, кнопкой ▶︎ Play |
| 3 | Ячейка 3 (установка) — 1–2 минуты |
| 4 | Ячейка 4 (модель) — 3–8 минут, это самое долгое |
| 5 | Ячейка 5 (запрос) — 1–3 минуты |

> ⏱ **Итого: ~10 минут.** Прогресс виден в выводе каждой ячейки.
> После перезапуска сессии ячейки 3 и 4 выполняются заново, но зависимости
> и модель остаются на диске Colab, поэтому повторно качать 4.7 ГБ не нужно.

---

## 🛠 Что внутри

| Ячейка | Что делает |
|---|---|
| 1 | Markdown: вы находитесь здесь |
| 2 | Клонирует/обновляет репозиторий, проверяет `commit` и `PARSER_VERSION` |
| 3 | Ставит зависимости, **включая готовый CUDA-wheel** llama-cpp-python |
| 4 | Проверяет GPU и объём видеопамяти |
| 5 | Скачивает и загружает Qwen 2.5 7B в видеопамять |
| 6 | Отправляет тестовый запрос учёного агенту |
| 7 | Показывает построенные графики прямо в ноутбуке |

---

## ⚠️ Три подводных камня, которые уже обойдены

**1. Компиляция CUDA.** На PyPI у `llama-cpp-python` лежит только sdist —
исходник. Обычный `pip install` компилирует CUDA 20–40 минут и часто
падает по таймауту. В ячейке 3 ставится **готовый wheel** с CUDA
(`cu121`, `manylinux_2_35` — совместим с Ubuntu 22.04 в Colab).

**2. Модель разбита на два файла.** Upstream не отдаёт единый
`qwen2.5-7b-instruct-q4_k_m.gguf` — только шарды
`...-00001-of-00002.gguf` и `...-00002-of-00002.gguf`. Прямой
`hf_hub_download` одного файла вернёт 404. Загрузкой занимается
`agent_core.download_model()`, который знает про шарды.

**3. Контекст 4096 не хватает.** Системный промпт плюс шесть схем
инструментов — это уже ~2000 токенов, а ответы инструментов занимают
ещё по 500–1300. Ноутбук использует `n_ctx=8192`.

In [ ]:
# ── Ячейка 2: репозиторий (и защита от устаревшего кода) ──
# ВРАЖНО: почему вы видите маленький ответ,
# а не агент? См. ниже: накончение и проверка школьки.
import os, sys, glob, shutil, subprocess

REPO = "https://github.com/IgorCim/geomagnetic-agent.git"
DIR  = "/content/geomagnetic_agent"
MODULES = ("agent_core", "geomag_analyzer", "geomag_plotter", "intermagnet_loader")

if os.path.isdir(os.path.join(DIR, ".git")):
    r = subprocess.run(["git", "-C", DIR, "pull", "--ff-only"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # Не глотаем: пусть упал весь старый код молча
        # будет ведся старый агент. Скажите выбранный
        # вариант или перезапустите сессию.
        print(r.stdout, r.stderr)
        shutil.rmtree(DIR, ignore_errors=True)
        subprocess.run(["git", "clone", REPO, DIR], check=True)
        print("ℹ️  Пуль не сработал — клон снова из GitHub.")
    else:
        print("ℹ️  git pull:", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "ok")
else:
    subprocess.run(["git", "clone", REPO, DIR], check=True)

# Старый байткод из предыдущего запуска сохранился бы.
for pc in glob.glob(os.path.join(DIR, "__pycache__")):
    shutil.rmtree(pc, ignore_errors=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ГЛАВНАЯ ПРИЧИНА СТАРОГО КОДА:
# Python кэширует уже импортированный модуль в sys.modules.
# Без удаления прежний вернии, ячейки 5-6 получают
# вчераонего агента, но старый parse_tool_calls без отладки.
dropped = [m for m in MODULES if m in sys.modules]
for m in dropped:
    del sys.modules[m]

import agent_core
sha = subprocess.run(["git", "-C", DIR, "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()

print("✅ Код:", os.getcwd())
print("   commit          :", sha)
print("   PARSER_VERSION  :", agent_core.PARSER_VERSION)
if dropped:
    print("   очищено старых модулей из кэша:", ", ".join(dropped))
if agent_core.PARSER_VERSION != "bulletproof-3":
    raise SystemExit("❌ ВЕРСИЯ КОДА УСТАРЕЛА! "
                     "Перезапустите сессию: "
                     "Среда выполните ячейку 2 сначала.")
print("   ✓ код актуален, парсер совпадает")


In [ ]:
# ── Ячейка 3: зависимости ────────────────────────────────────────────────
# 1. llama-cpp-python С ГОТОВЫМ CUDA. На PyPI только sdist -> обычный pip
#    компилирует CUDA 20-40 минут и часто падает. Ставим wheel по прямой
#    ссылке. --force-reinstall обязателен: на образе Colab уже может лежать
#    CPU-сборка той же версии, и pip её не тронет.
!pip install -q --force-reinstall --no-deps https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu121/llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl

# 2. Зависимости самого wheel
!pip install -q "jinja2>=3.1.2" "diskcache>=5.6" "numpy>=1.26"

# 3. Данные и визуализация. pyarrow опционален: без него кэш идёт в CSV.
!pip install -q pandas plotly requests huggingface_hub pyarrow

import llama_cpp
print("✅ Зависимости установлены! llama-cpp-python", llama_cpp.__version__)

In [ ]:
# ── Ячейка 4: проверка оборудования ──────────────────────────────────────
import torch
from llama_cpp import llama_cpp as _lc

print("=" * 52)
print("  ПРОВЕРКА ОБОРУДОВАНИЯ")
print("=" * 52)

gpu_ok = False
if torch.cuda.is_available():
    print(f"✅ GPU доступен: {torch.cuda.get_device_name(0)}")
    print(f"✅ Память GPU  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    gpu_ok = True
else:
    print("❌ torch не видит GPU.")

# Это важнее torch: именно она решает, умеет ли llama.cpp работать на GPU.
if _lc.llama_supports_gpu_offload():
    print("✅ llama.cpp: выгрузка на GPU поддерживается")
else:
    print("❌ llama.cpp БЕЗ CUDA — стоит CPU-сборка.")
    print("   Перезапустите ячейку 3: возможно, pip не переустановил wheel.")
    gpu_ok = False

if not gpu_ok:
    raise SystemExit(
        "Остановитесь и включите GPU: Среда выполнения → "
        "Сменить среду выполнения → GPU (T4), затем выполните ячейки 3-4 заново."
    )

print("=" * 52)
print("Модель Q4_K_M занимает ~4.7 ГБ из 16 ГБ — VRAM хватает с запасом.")

In [ ]:
# ── Ячейка 5: загрузка "Мозга" (Qwen 2.5 7B Instruct, Q4_K_M) ────────────
# ВАЖНО: не используем hf_hub_download с именем одного файла — модели
# qwen2.5-7b-instruct-q4_k_m.gguf НЕ существует, она разбита на 2 шарда.
# agent_core.download_model() это знает.
#
# n_ctx=8192, а не 4096: системный промпт + 6 схем инструментов — уже
# ~2000 токенов, плюс ответы инструментов по 500-1300 токенов каждый.
import agent_core

print(f"Репозиторий : {agent_core.MODEL_REPO}")
print(f"Квантование : {agent_core.MODEL_STEM}  (2 шарда, ~4.7 ГБ)")
print(f"Папка       : {agent_core.MODELS_DIR}")
print("⏳ Скачиваю модель, это 3-8 минут...")

brain = agent_core.load_brain(n_gpu_layers=-1, n_ctx=8192, verbose=True)

if agent_core.is_error(brain):
    raise SystemExit(
        f"\n❌ Мозг не загрузился.\n"
        f"   код    : {brain['error']}\n"
        f"   причина: {brain['message']}"
        f"\n   совет  : {brain.get('hint', 'проверьте ячейку 3 (wheel) и ячейку 4 (GPU)')}"
    )

print("\n✅ Мозг (Qwen 2.5 7B) успешно загружен в GPU!")
print("   Объект brain — это обёртка LlamaBrain, умеющая .chat(messages, tools).")

In [ ]:
# ── Ячейка 6: тестовый запрос учёного ─────────────────────────────────────
from agent_core import run_agent, DEMO_QUERY

QUERY = ("Скачай данные по станции Иркутск (код IRT) за 10 сентября 2024 года. "
         "Посчитай компоненту H и построй график.")

print("🚀 Отправляю запрос агенту...")
print("   " + QUERY)

# brain — обёртка из ячейки 5. Сырой Llama сюда передавать нельзя:
# run_agent ждёт объект с методом .chat(messages, tools).
result = run_agent(QUERY, brain=brain)

print("\n" + "=" * 52)
print("  ЖУРНАЛ ВЫЗОВОВ ИНСТРУМЕНТОВ")
print("=" * 52)
for i, step in enumerate(result.get("tool_calls", []), 1):
    mark = "ok " if step.get("ok") else "ОШИБКА"
    print(f"{i}. [{mark}] {step['tool']}")
    if step.get("error"):
        print(f"      {step['error']}")
        if step.get("reason"):
            print(f"      причина : {step['reason']}")
        if step.get("received"):
            print(f"      получено: {step['received']}")
        if step.get("hint"):
            print(f"      совет   : {step['hint']}")
        if step.get("message"):
            print(f"      сообщ   : {step['message']}")
    if step.get("plot"):
        print(f"      график -> {step['plot']}")

print("\n" + "=" * 52)
print("  ОТВЕТ АГЕНТА")
print("=" * 52)
print(result.get("text", "(пусто)"))

print(f"\nРаундов: {result.get('rounds')}   Причина остановки: {result.get('stop_reason')}")
if result.get("plots"):
    print("\n✅ Сгенерированные графики:")
    for p in result["plots"]:
        print(f"   - {p}")

In [ ]:
# ── Ячейка 7: смотрим графики прямо в ноутбуке ────────────────────────────
import os
from IPython.display import IFrame, display, HTML

plots = result.get("plots", [])
if not plots:
    print("Графиков нет — проверьте ошибки в ячейке 6.")
else:
    for p in plots:
        print("показываю:", p)
        # Файл самодостаточный (Plotly вшит внутрь), поэтому открывается офлайн.
        display(IFrame(src=p, width="100%", height=560))

    # Скачать архивом на свой компьютер:
    import shutil
    shutil.make_archive("/content/geomagnetic_plots", "zip", "plots")
    print("\nАрхив с графиками: /content/geomagnetic_plots.zip")
    print("Скачать: панель слева -> Files -> /content/geomagnetic_plots.zip")